In [ ]:
# FitAI — Nutrition Dataset Pipeline (Notebook 2)

# CELL 1 — Install dependencies
!pip install google-generativeai requests beautifulsoup4 pandas tqdm --quiet

# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/FitAI/datasets/nutrition'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Saving to: {SAVE_DIR}")

# CELL 3 — API Configuration
from google.colab import userdata
import google.generativeai as genai
import requests, json, time, re
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemma-3-27b-it')
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; FitAI-research-bot/1.0)'}
print("APIs ready")

# CELL 4 — Country List
COUNTRIES = [
  "Nigeria", "Kenya", "South Africa", "Ghana", "Ethiopia", "Tanzania",
  "Uganda", "Egypt", "Morocco", "Senegal", "Cameroon", "Zimbabwe",
  "Zambia", "Rwanda", "Ivory Coast", "Angola", "Mozambique",
  "United States", "Brazil", "Mexico", "Colombia", "Argentina",
  "Peru", "Chile", "Venezuela", "Ecuador", "Canada",
  "United Kingdom", "Germany", "France", "Italy", "Spain",
  "Portugal", "Netherlands", "Poland", "Ukraine", "Turkey",
  "India", "China", "Indonesia", "Pakistan", "Bangladesh",
  "Philippines", "Vietnam", "Thailand", "Malaysia", "Japan",
  "South Korea", "Saudi Arabia", "UAE", "Iran",
  "Australia", "New Zealand",
]
print(f"Total countries: {len(COUNTRIES)}")

# CELL 5 — Universal Foods Baseline
UNIVERSAL_FOODS = [
  "eggs", "rice", "chicken breast", "bread", "pasta",
  "potatoes", "onions", "tomatoes", "cooking oil", "milk",
  "bananas", "apples", "oranges", "carrots", "cabbage",
  "beef", "fish", "lentils", "beans", "oats",
  "sugar", "salt", "butter", "cheese", "yogurt",
  "garlic", "ginger", "spinach", "sweet potatoes", "peanuts",
]

UNIVERSAL_NUTRITION = {
  "eggs":           {"calories": 155, "protein_g": 13, "carbs_g": 1,  "fat_g": 11, "unit": "100g"},
  "rice":           {"calories": 130, "protein_g": 3,  "carbs_g": 28, "fat_g": 0,  "unit": "100g"},
  "chicken breast": {"calories": 165, "protein_g": 31, "carbs_g": 0,  "fat_g": 4,  "unit": "100g"},
  "bread":          {"calories": 265, "protein_g": 9,  "carbs_g": 49, "fat_g": 3,  "unit": "100g"},
  "pasta":          {"calories": 131, "protein_g": 5,  "carbs_g": 25, "fat_g": 1,  "unit": "100g"},
  "bananas":        {"calories": 89,  "protein_g": 1,  "carbs_g": 23, "fat_g": 0,  "unit": "100g"},
  "lentils":        {"calories": 116, "protein_g": 9,  "carbs_g": 20, "fat_g": 0,  "unit": "100g"},
  "oats":           {"calories": 389, "protein_g": 17, "carbs_g": 66, "fat_g": 7,  "unit": "100g"},
}
print(f"Universal foods defined: {len(UNIVERSAL_FOODS)}")

# CELL 6 — Numbeo Scraper
def scrape_numbeo(country):
    url = f"https://www.numbeo.com/food-prices/country_result.jsp?country={country.replace(' ', '+')}"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        table = soup.find('table', {'class': 'data_wide_table'})
        if not table:
            return {}
        prices = {}
        for row in table.find_all('tr')[1:]:
            cols = row.find_all('td')
            if len(cols) >= 2:
                item = cols[0].text.strip().lower()
                price = cols[1].text.strip().replace(',', '.')
                try:
                    prices[item] = float(re.sub(r'[^\d.]', '', price))
                except:
                    pass
        return prices
    except Exception as e:
        print(f"  Numbeo error ({country}): {e}")
        return {}

# CELL 7 — WFP VAM Scraper
def fetch_wfp_data(country):
    url = f"https://api.vam.wfp.org/foodprices/api/v1/country-data?country={country.replace(' ', '%20')}&output=json"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        data = r.json()
        prices = {}
        for item in data.get('data', []):
            commodity = item.get('commodity', '').lower()
            price = item.get('price')
            currency = item.get('currency', 'USD')
            if commodity and price:
                prices[commodity] = {'price': price, 'currency': currency, 'source': 'wfp'}
        return prices
    except Exception as e:
        print(f"  WFP error ({country}): {e}")
        return {}

# CELL 8 — FAO Baseline
def fetch_fao_commodities():
    url = "https://www.fao.org/giews/food-prices/food-index/en/"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        commodities = {}
        for row in soup.find_all('tr'):
            cols = row.find_all('td')
            if len(cols) >= 2:
                name = cols[0].text.strip().lower()
                val = cols[1].text.strip()
                try:
                    commodities[name] = float(re.sub(r'[^\[d.]', '', val))
                except:
                    pass
        return commodities
    except Exception as e:
        print(f"  FAO error: {e}")
        return {}

fao_baseline = fetch_fao_commodities()
print(f"FAO baseline commodities fetched: {len(fao_baseline)}")

# CELL 9 — Gemma Local Foods Extractor
LOCAL_FOODS_PROMPT = """
You are a food and nutrition expert with knowledge of every country's cuisine.

For the country: {country}

Return a JSON array of local foods commonly eaten there that are NOT in this universal list:
{universal_list}

For each local food include:
- food_name: string (in English)
- local_name: string (name used locally)
- food_type: one of [protein, carbohydrate, vegetable, fruit, fat, dairy, beverage]
- calories_per_100g: number
- protein_g: number
- carbs_g: number
- fat_g: number
- typical_meal: one of [breakfast, lunch, dinner, snack, any]
- budget_tier: one of [low, medium, high]
- availability: one of [widely_available, moderately_available, seasonal]
- dietary_tags: array from [vegetarian, vegan, gluten_free, halal, kosher]

Return ONLY a valid JSON array. No explanation, no markdown, no preamble.
"""

def get_local_foods(country):
    prompt = LOCAL_FOODS_PROMPT.format(country=country, universal_list=', '.join(UNIVERSAL_FOODS))
    try:
        response = model.generate_content(prompt)
        raw = response.text.strip()
        raw = re.sub(r'```json|```', '', raw).strip()
        return json.loads(raw)
    except Exception as e:
        print(f"  Gemma local foods error ({country}): {e}")
        return []

print("Local foods extractor ready")

# CELL 10 — Main Pipeline Loop with Resume Support
checkpoint_path = f"{SAVE_DIR}/checkpoint_nutrition.json"
checkpoint_done_path = f"{SAVE_DIR}/checkpoint_countries_done.json"

if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        all_food_records = json.load(f)
    print(f"Resumed: {len(all_food_records)} records")
else:
    all_food_records = []

if os.path.exists(checkpoint_done_path):
    with open(checkpoint_done_path) as f:
        countries_done = set(json.load(f))
    print(f"Skipping {len(countries_done)} completed countries")
else:
    countries_done = set()

for country in tqdm(COUNTRIES, desc="Processing countries"):
    if country in countries_done:
        continue
    print(f"\n Processing: {country}")
    local_foods = get_local_foods(country)
    time.sleep(4)
    numbeo_prices = scrape_numbeo(country)
    time.sleep(2)
    wfp_prices = fetch_wfp_data(country)
    time.sleep(2)

    for food in UNIVERSAL_FOODS:
        nutrition = UNIVERSAL_NUTRITION.get(food, {})
        price_usd = numbeo_prices.get(food) or wfp_prices.get(food, {}).get('price') or fao_baseline.get(food)
        all_food_records.append({
            'country': country, 'food_name': food, 'food_type': 'universal',
            'is_local': False, 'is_universal': True, 'availability_confirmed': True,
            'price_usd_per_kg': price_usd, 'price_source': 'numbeo+wfp+fao',
            'calories_per_100g': nutrition.get('calories'), 'protein_g': nutrition.get('protein_g'),
            'carbs_g': nutrition.get('carbs_g'), 'fat_g': nutrition.get('fat_g'),
            'budget_tier': 'low' if price_usd and price_usd < 2 else 'medium' if price_usd and price_usd < 5 else 'high',
            'dietary_tags': [], 'typical_meal': 'any',
        })

    for food in local_foods:
        food_name = food.get('food_name', '').lower()
        price_usd = numbeo_prices.get(food_name) or wfp_prices.get(food_name, {}).get('price') or None
        all_food_records.append({
            'country': country, 'food_name': food_name, 'local_name': food.get('local_name', ''),
            'food_type': food.get('food_type', ''), 'is_local': True, 'is_universal': False,
            'availability_confirmed': price_usd is not None, 'price_usd_per_kg': price_usd,
            'price_source': 'numbeo+wfp' if price_usd else 'gemma_estimated',
            'calories_per_100g': food.get('calories_per_100g'), 'protein_g': food.get('protein_g'),
            'carbs_g': food.get('carbs_g'), 'fat_g': food.get('fat_g'),
            'budget_tier': food.get('budget_tier', 'medium'), 'dietary_tags': food.get('dietary_tags', []),
            'typical_meal': food.get('typical_meal', 'any'),
        })

    countries_done.add(country)
    with open(checkpoint_path, 'w') as f:
        json.dump(all_food_records, f, indent=2)
    with open(checkpoint_done_path, 'w') as f:
        json.dump(list(countries_done), f, indent=2)
    print(f"  Done. Total records so far: {len(all_food_records)}")

print(f"\nPipeline complete! Total food records: {len(all_food_records)}")

# CELL 11 — Build Fine-tuning Nutrition Dataset
finetune_nutrition = []
df = pd.DataFrame(all_food_records)

for country in COUNTRIES:
    country_foods = df[df['country'] == country]
    if country_foods.empty:
        continue
    affordable = country_foods[country_foods['budget_tier'] == 'low']['food_name'].tolist()[:10]
    finetune_nutrition.append({
        "instruction": "Create a healthy meal plan for someone on a low budget.",
        "input": f"Country: {country}, Budget: low, Meals per day: 3",
        "output": f"Here is a budget friendly meal plan for {country}: Breakfast: {affordable[0] if len(affordable) > 0 else 'eggs'} with {affordable[1] if len(affordable) > 1 else 'bread'}. Lunch: {affordable[2] if len(affordable) > 2 else 'rice'} with {affordable[3] if len(affordable) > 3 else 'beans'}. Dinner: {affordable[4] if len(affordable) > 4 else 'chicken'} with {affordable[5] if len(affordable) > 5 else 'vegetables'}."
    })
    local = country_foods[country_foods['is_local'] == True]['food_name'].tolist()[:5]
    finetune_nutrition.append({
        "instruction": "What local foods are available in this country?",
        "input": f"Country: {country}",
        "output": f"Common local foods in {country} include: {', '.join(local)}." if local else f"{country} primarily uses universal staple foods."
    })
    high_protein = country_foods[country_foods['protein_g'] >= 15]['food_name'].tolist()[:5]
    finetune_nutrition.append({
        "instruction": "Recommend high protein foods available in this country.",
        "input": f"Country: {country}, Goal: muscle building",
        "output": f"For muscle building in {country}, focus on: {', '.join(high_protein)}." if high_protein else f"Focus on eggs, chicken, and lentils as protein sources in {country}."
    })

print(f"Nutrition fine-tuning pairs: {len(finetune_nutrition)}")

# CELL 12 — Save All Datasets to Google Drive
csv_path = f"{SAVE_DIR}/unified_food_database.csv"
df.to_csv(csv_path, index=False)
print(f"Unified food database saved: {csv_path}")

raw_path = f"{SAVE_DIR}/raw_food_records.json"
with open(raw_path, 'w') as f:
    json.dump(all_food_records, f, indent=2)
print(f"Raw records saved: {raw_path}")

ft_path = f"{SAVE_DIR}/finetune_nutrition_dataset.json"
with open(ft_path, 'w') as f:
    json.dump(finetune_nutrition, f, indent=2)
print(f"Fine-tuning dataset saved: {ft_path}")

print(f"\n--- Dataset Summary ---")
print(f"Total food records:        {len(all_food_records)}")
print(f"Countries covered:         {df['country'].nunique()}")
print(f"Unique foods:              {df['food_name'].nunique()}")
print(f"Local foods:               {df[df['is_local']==True].shape[0]}")
print(f"Universal foods:           {df[df['is_universal']==True].shape[0]}")
print(f"Availability confirmed:    {df[df['availability_confirmed']==True].shape[0]}")
print(f"Fine-tuning pairs:         {len(finetune_nutrition)}")
print(f"\nAll files saved to Google Drive!")